In [0]:
# Extract

import glob
import shutil
import os
import json
import xml.etree.ElementTree as ET
from datetime import datetime
import pandas as pd

# Based location
VOLUME_DIR = "/Volumes/etl_meridian_project/etl_meridianschema/etl_meridianvolume"

# Sub-folders for my files
RAW_DIR = f"{VOLUME_DIR}/raw"
OUTPUT_DIR = f"{VOLUME_DIR}/processed"
LOG_DIR = f"{VOLUME_DIR}/logs"

# Save files
TARGET_FILE = f"{OUTPUT_DIR}/sales_consoldated.csv"
LOG_FILE = f"{LOG_DIR}/etl_log.txt"

# Create the folders if they don't exist yet
for folder in [RAW_DIR, OUTPUT_DIR, LOG_DIR]:
    os.makedirs(folder, exist_ok=True)
# Log function to keep track of the logging process.
def log_progress(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {message}"
    with open(LOG_FILE, 'a') as file:
        file.write(line + '\n')
    print(line)
def extract_csv(path):
    # Store POS sales: already flat, read as text
    return pd.read_csv(path, dtype=str)


def extract_json(path):
    # Web orders: nested JSON -> one row per item
    with open(path) as f:
        orders = [json.loads(line) for line in f]

    return pd.json_normalize(
        orders,
        record_path="items",
        meta=["order_id", "order_ts", "storefront", "currency", "status",
              ["customer", "customer_id"], ["customer", "country"]],
        sep="_",
    )


def extract_xml(path):
    # Product catalog: walk each <product> tag
    root = ET.parse(path).getroot()
    rows = []
    for p in root.findall("product"):
        rows.append({
            "product_id": p.get("id"),
            "name":       p.findtext("name"),
            "category":   p.findtext("category"),
            "list_price": p.findtext("pricing/list_price"),
            "unit_cost":  p.findtext("pricing/unit_cost"),
        })
    return pd.DataFrame(rows)


def extract_excel(path):
    # Finance workbook: reads all sheets -> {sheet_name: DataFrame}
    return pd.read_excel(path, sheet_name=None, dtype=str)



# Redefine log_progress to work on UC Volumes (append mode not supported)
def log_progress(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {message}"
    try:
        existing = open(LOG_FILE, 'r').read()
    except FileNotFoundError:
        existing = ""
    with open(LOG_FILE, 'w') as file:
        file.write(existing + line + '\n')
    print(line)


def extract():
    raw = {}

    for file in glob.glob(f"{RAW_DIR}/*.csv"):
        raw["store_sales"] = extract_csv(file)
        log_progress(f"Extracted CSV: {os.path.basename(file)}")

    for file in glob.glob(f"{RAW_DIR}/*.json"):
        raw["web_orders"] = extract_json(file)
        log_progress(f"Extracted JSON: {os.path.basename(file)}")

    for file in glob.glob(f"{RAW_DIR}/*.xml"):
        raw["products"] = extract_xml(file)
        log_progress(f"Extracted XML: {os.path.basename(file)}")

    for file in glob.glob(f"{RAW_DIR}/*.xlsx"):
        raw["finance"] = extract_excel(file)
        log_progress(f"Extracted Excel: {os.path.basename(file)}")

    return raw


log_progress("Extract phase Started")
raw = extract()
log_progress("Extract phase Ended")


for name, data in raw.items():
    if isinstance(data, dict):                     # Excel: a dict of sheets
        for sheet, df in data.items():
            print(f"{name}/{sheet}: {df.shape}")
    else:
        print(f"{name}: {data.shape}")
display(raw["finance"]["fx_rates"].head(20))
# Save the data for further processing
BRONZE_DIR = f"{VOLUME_DIR}/bronze"
os.makedirs(BRONZE_DIR, exist_ok=True)

for name, data in raw.items():
    if isinstance(data, dict):                    # Excel: save each sheet separately
        for sheet, df in data.items():
            df.astype("string").to_parquet(f"{BRONZE_DIR}/{sheet}.parquet")
            log_progress(f"Saved bronze: {sheet}")
    else:
        data.astype("string").to_parquet(f"{BRONZE_DIR}/{name}.parquet")
        log_progress(f"Saved bronze: {name}")

In [0]:
# Transform

import glob
import shutil
import os
import json
import xml.etree.ElementTree as ET
from datetime import datetime
import pandas as pd

# Based location
VOLUME_DIR = "/Volumes/etl_meridian_project/etl_meridianschema/etl_meridianvolume"

# Sub-folders for my files
RAW_DIR = f"{VOLUME_DIR}/raw"
OUTPUT_DIR = f"{VOLUME_DIR}/processed"
LOG_DIR = f"{VOLUME_DIR}/logs"

# Save files
TARGET_FILE = f"{OUTPUT_DIR}/sales_consoldated.csv"
LOG_FILE = f"{LOG_DIR}/etl_log.txt"
BRONZE_DIR = f"{VOLUME_DIR}/bronze"
os.makedirs(BRONZE_DIR, exist_ok=True)


VOLUME_DIR = "/Volumes/etl_meridian_project/etl_meridianschema/etl_meridianvolume"
BRONZE_DIR = f"{VOLUME_DIR}/bronze"

bronze = {}
for file in glob.glob(f"{BRONZE_DIR}/*.parquet"):
    name = os.path.basename(file).replace(".parquet", "")
    bronze[name] = pd.read_parquet(file)

for name, df in bronze.items():
    print(name, df.shape)
# Log function to keep track of the logging process.
def log_progress(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {message}"
    with open(LOG_FILE, 'w') as file:
        file.write(line + '\n')
    print(line)
# ---------------------------------------------------------------- store sales (CSV)

def clean_store_sales(df, locations):
    df = df.copy()

    # Tidy IDs first, so duplicates become truly identical
    df["location_id"] = df["location_id"].str.strip().str.upper()
    df["product_id"]  = df["product_id"].str.strip()

    # POS resends: exact duplicate rows
    df = df.drop_duplicates()

    # Numbers: Berlin writes "94,99"
    df["unit_price"]   = pd.to_numeric(df["unit_price"].str.replace(",", "."), errors="coerce")
    df["quantity"]     = pd.to_numeric(df["quantity"])
    df["discount_pct"] = pd.to_numeric(df["discount_pct"])

    # Dates: normal format first, then Berlin's format fills the gaps
    dt = df["transaction_datetime"]
    df["transaction_datetime"] = (
        pd.to_datetime(dt, format="%Y-%m-%d %H:%M:%S", errors="coerce")
          .fillna(pd.to_datetime(dt, format="%d.%m.%Y %H:%M", errors="coerce"))
    )

    # Missing currency: each store has one currency, so look it up in locations
    store_currency = locations.set_index("location_id")["currency"]
    df["currency"] = df["currency"].fillna(df["location_id"].map(store_currency))

    # Missing payment method: label it
    df["payment_method"] = df["payment_method"].fillna("unknown")

    # No price = can't compute revenue: drop and log
    missing_price = df["unit_price"].isna().sum()
    df = df.dropna(subset=["unit_price"])
    log_progress(f"store_sales: dropped {missing_price} rows with no price")

    return df


# ---------------------------------------------------------------- web orders (JSON)

COUNTRY_CURRENCY = {"US": "USD", "CA": "CAD", "GB": "GBP", "DE": "EUR", "FR": "EUR", "NL": "EUR"}

def clean_web_orders(df):
    df = df.copy()

    # Remove QA test orders
    df = df[~df["customer_customer_id"].str.startswith("TEST")]

    # Event log: a refunded order appears twice (placed, then updated).
    # Keep the LAST event per order item = its latest status. Also removes exact duplicates.
    df = df.drop_duplicates(subset=["order_id", "product_id"], keep="last")

    # Numbers: some qty values were sent as text
    df["qty"]           = pd.to_numeric(df["qty"])
    df["unit_price"]    = pd.to_numeric(df["unit_price"])
    df["line_discount"] = pd.to_numeric(df["line_discount"])

    # Timestamps: most are ISO text, some are epoch milliseconds
    ts = df["order_ts"]
    is_epoch = ts.str.isdigit()
    df["order_ts"] = pd.to_datetime(ts.where(~is_epoch), utc=True, errors="coerce")
    df.loc[is_epoch, "order_ts"] = pd.to_datetime(ts[is_epoch].astype("int64"), unit="ms", utc=True)

    # Missing currency: the customer's country tells us what they paid in
    df["currency"] = df["currency"].fillna(df["customer_country"].map(COUNTRY_CURRENCY))

    return df


# ---------------------------------------------------------------- products (XML)

def clean_products(df):
    df = df.copy()

    df["product_id"] = df["product_id"].str.strip()

    # Category typed inconsistently: " outdoor ", "OUTDOOR" -> "Outdoor"
    df["category"] = df["category"].str.strip().str.title()

    # Prices: some costs look like "$12.30", some are empty
    df["list_price"] = pd.to_numeric(df["list_price"])
    df["unit_cost"]  = pd.to_numeric(df["unit_cost"].str.replace("$", ""), errors="coerce")

    # Same product exported twice: keep one, log how many
    dupes = df["product_id"].duplicated().sum()
    df = df.drop_duplicates(subset="product_id", keep="last")
    log_progress(f"products: removed {dupes} duplicate product records")

    return df


# ---------------------------------------------------------------- locations (Excel)

def clean_locations(df):
    df = df.copy()
    df["location_id"]        = df["location_id"].str.strip().str.upper()
    df["store_size_sqm"]     = pd.to_numeric(df["store_size_sqm"])
    df["monthly_rent_local"] = pd.to_numeric(df["monthly_rent_local"])
    df["opening_date"]       = pd.to_datetime(df["opening_date"])
    return df


# ---------------------------------------------------------------- fx rates (Excel)

def clean_fx_rates(df):
    df = df.copy()

    df["currency"] = df["currency"].str.strip().str.upper()

    # Rates typed as "1,0912" -> 1.0912
    df["rate_to_usd"] = pd.to_numeric(df["rate_to_usd"].str.replace(",", "."), errors="coerce")

    # Dates: real Excel dates, plus some typed as "07/15/2026"
    d = df["rate_date"]
    df["rate_date"] = (
        pd.to_datetime(d, format="%Y-%m-%d %H:%M:%S", errors="coerce")
          .fillna(pd.to_datetime(d, format="%m/%d/%Y", errors="coerce"))
    )

    # Fill weekends and missing days: every day gets the last known rate
    all_days = pd.date_range(df["rate_date"].min(), df["rate_date"].max(), freq="D")
    df = (
        df.dropna(subset=["rate_to_usd"])
          .pivot_table(index="rate_date", columns="currency", values="rate_to_usd")
          .reindex(all_days)
          .ffill()
          .rename_axis("rate_date")
          .reset_index()
          .melt(id_vars="rate_date", var_name="currency", value_name="rate_to_usd")
    )
    return df



# ---------------------------------------------------------------- transform()

def transform(bronze):
    silver = {}
    silver["locations"]   = clean_locations(bronze["locations"])
    silver["fx_rates"]    = clean_fx_rates(bronze["fx_rates"])
    silver["products"]    = clean_products(bronze["products"])
    silver["store_sales"] = clean_store_sales(bronze["store_sales"], silver["locations"])
    silver["web_orders"]  = clean_web_orders(bronze["web_orders"])

    for name, df in silver.items():
        log_progress(f"Cleaned {name}: {len(bronze[name])} -> {len(df)} rows")
    return silver
log_progress("Transform phase Started")
silver = transform(bronze)
log_progress("Transform phase Ended")

SILVER_DIR = f"{VOLUME_DIR}/silver"
os.makedirs(SILVER_DIR, exist_ok=True)

for name, df in silver.items():
    df.to_parquet(f"{SILVER_DIR}/{name}.parquet")
    log_progress(f"Saved silver: {name}")

In [0]:
# Load
import glob
import shutil
import os
import json
import xml.etree.ElementTree as ET
from datetime import datetime
import pandas as pd
import numpy as np
# Based location
VOLUME_DIR = "/Volumes/etl_meridian_project/etl_meridianschema/etl_meridianvolume"

# Sub-folders for my files
RAW_DIR = f"{VOLUME_DIR}/raw"
OUTPUT_DIR = f"{VOLUME_DIR}/processed"
LOG_DIR = f"{VOLUME_DIR}/logs"

# Save files
TARGET_FILE = f"{OUTPUT_DIR}/sales_consoldated.csv"
LOG_FILE = f"{LOG_DIR}/etl_log.txt"
BRONZE_DIR = f"{VOLUME_DIR}/bronze"
os.makedirs(BRONZE_DIR, exist_ok=True)


SILVER_DIR  = f"{VOLUME_DIR}/silver"
TARGET_FILE = f"{VOLUME_DIR}/processed/sales_consolidated.csv"

silver = {}
for file in glob.glob(f"{SILVER_DIR}/*.parquet"):
    name = os.path.basename(file).replace(".parquet", "")
    silver[name] = pd.read_parquet(file)
# Log function to keep track of the logging process.
def log_progress(message):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {message}"
    with open(LOG_FILE, 'w') as file:
        file.write(line + '\n')
    print(line)
# ---------------------------------------------------------------- 1. standardize

def standardize_store_sales(df):
    return pd.DataFrame({
        "sale_id":          df["transaction_id"] + "-" + df["line_no"],
        "sale_date":        df["transaction_datetime"].dt.normalize(),
        "location_id":      df["location_id"],
        "product_id":       df["product_id"],
        "quantity":         df["quantity"],
        "unit_price_local": df["unit_price"],
        "discount_rate":    df["discount_pct"] / 100,
        "currency":         df["currency"],
        "status":           np.where(df["quantity"] < 0, "returned", "completed"),
    })


def standardize_web_orders(df):
    return pd.DataFrame({
        "sale_id":          df["order_id"] + "-" + df["product_id"],
        "sale_date":        df["order_ts"].dt.tz_convert(None).dt.normalize(),
        "location_id":      df["storefront"],
        "product_id":       df["product_id"],
        "quantity":         df["qty"],
        "unit_price_local": df["unit_price"],
        "discount_rate":    df["line_discount"],
        "currency":         df["currency"],
        "status":           df["status"],
        "country":          df["customer_country"],   # where the customer is
    })


# ---------------------------------------------------------------- 2. combine

def combine(store, web):
    sales = pd.concat([store, web], ignore_index=True)
    log_progress(f"Combined: {len(store)} store + {len(web)} web = {len(sales)} sales")
    return sales

 #---------------------------------------------------------------- 3. enrich

def enrich(sales, products, locations, fx_rates):
    sales = sales.merge(
        locations[["location_id", "channel", "region", "country"]]
            .rename(columns={"country": "location_country"}),
        on="location_id", how="left")
    # Store sales: country = the store's country. Web sales already have the customer's country.
    sales["country"] = sales["country"].fillna(sales["location_country"])

    sales = sales.merge(
        products[["product_id", "name", "category", "unit_cost"]],
        on="product_id", how="left")

    sales = sales.merge(
        fx_rates.rename(columns={"rate_date": "sale_date", "rate_to_usd": "fx_rate"}),
        on=["sale_date", "currency"], how="left")

    unknown = sales["name"].isna().sum()
    sales["name"] = sales["name"].fillna("Unknown product")
    sales["category"] = sales["category"].fillna("Unknown")
    log_progress(f"Enrich: {unknown} sales with a product_id not in the catalog")
    return sales


# ---------------------------------------------------------------- 4. calculate

def calculate(sales):
    # Business rule: refunded and cancelled orders earn nothing
    counts = sales["status"].isin(["completed", "returned"])

    revenue = sales["quantity"] * sales["unit_price_local"] * (1 - sales["discount_rate"]) * sales["fx_rate"]
    cost    = sales["quantity"] * sales["unit_cost"]

    sales["revenue_usd"] = np.where(counts, revenue, 0).round(2)
    sales["cost_usd"]    = np.where(counts, cost, 0).round(2)
    sales["profit_usd"]  = (sales["revenue_usd"] - sales["cost_usd"]).round(2)
    return sales


# ---------------------------------------------------------------- 5. final shape

GOLD_COLUMNS = [
    "sale_id", "sale_date", "channel", "region", "country", "location_id",
    "product_id", "name", "category", "status", "quantity",
    "currency", "unit_price_local", "fx_rate",
    "revenue_usd", "cost_usd", "profit_usd",
]

def build_gold(silver):
    store = standardize_store_sales(silver["store_sales"])
    web   = standardize_web_orders(silver["web_orders"])
    sales = combine(store, web)
    sales = enrich(sales, silver["products"], silver["locations"], silver["fx_rates"])
    sales = calculate(sales)
    return sales[GOLD_COLUMNS].rename(columns={"name": "product_name"})
log_progress("Load phase Started")

gold = build_gold(silver)
gold.to_csv(TARGET_FILE, index=False)

log_progress(f"Saved {len(gold)} rows to {TARGET_FILE}")
log_progress("Load phase Ended")
log_progress("ETL Job Ended")

display(gold)